In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!nvidia-smi

Thu Apr 30 06:18:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [13]:
!ls -lh /content/drive/MyDrive/646final/models_download/

total 99G
-rw------- 1 root root 293M Apr 29 03:31 detailz-wan.safetensors
-rw------- 1 root root 451M Apr 28 00:13 edit_0928_lora_step40000.safetensors
-rw------- 1 root root 1.2G Apr 28 00:04 open-clip-xlm-roberta-large-vit-huge-14_visual_fp16.safetensors
-rw------- 1 root root 243M Apr 28 00:13 pig_qwen_image_vae_fp32-f16.gguf
-rw------- 1 root root  27G Apr 28 00:09 Qwen-Rapid-AIO-NSFW-v23.safetensors
-rw------- 1 root root  14G Apr 28 00:16 TurboWan2.2-I2V-A14B-high-720P-quant.pth
-rw------- 1 root root  14G Apr 28 00:19 TurboWan2.2-I2V-A14B-low-720P-quant.pth
-rw------- 1 root root 6.3G Apr 28 00:05 umt5-xxl-enc-fp8_e4m3fn.safetensors
-rw------- 1 root root 6.3G Apr 28 03:13 umt5_xxl_fp8_e4m3fn_scaled.safetensors
-rw------- 1 root root 243M Apr 28 01:31 Wan2.1_VAE.safetensors
-rw------- 1 root root  15G Apr 28 01:07 Wan2.2-I2V-A14B-HighNoise-Q8_0.gguf
-rw------- 1 root root  15G Apr 28 01:09 Wan2.2-I2V-A14B-LowNoise-Q8_0.gguf
-rw------- 1 root root 1.4G Apr 28 00:04 Wan2.2_VAE.sa

In [ ]:
import os


'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'

In [ ]:
import os
import subprocess
import time
import threading
import re
import socket
import shutil

from google.colab import drive
from google.colab import userdata

drive.mount("/content/drive")

# -------------------------------------------------------------------------
# 0. Colab global paths
# -------------------------------------------------------------------------
ROOT_DIR = "/content"
COMFY_DIR = os.path.join(ROOT_DIR, "ComfyUI")
MODELS_DIR = os.path.join(COMFY_DIR, "models")

MODEL_SOURCE_PATH = "/content/drive/MyDrive/646final/models_download"

# Optional Groq key from Colab secrets
try:
    groq_key = userdata.get("GROQ_API_KEY")
    if groq_key:
        os.environ["GROQ_API_KEY"] = groq_key
        print("Atara [Groq]: ✅ GROQ_API_KEY loaded from Colab secrets")
    else:
        print("Atara [Groq]: ⚠️ No GROQ_API_KEY found; Groq nodes may show only 'none'")
except Exception:
    print("Atara [Groq]: ⚠️ Could not load GROQ_API_KEY from Colab secrets")


def run_cmd(cmd, msg=None, allow_fail=False):
    if msg:
        print(f"Atara [System]: {msg}")
    try:
        subprocess.run(cmd, shell=True, check=True)
    except subprocess.CalledProcessError:
        if not allow_fail and "pkill" not in cmd:
            print(f"Atara [Warning]: command may have failed: {cmd}")


def clone_or_update(repo_url, target_dir, name):
    if not os.path.exists(target_dir):
        print(f"Atara [Builder]: Cloning {name}...")
        run_cmd(f"git clone {repo_url} {target_dir}")
    else:
        print(f"Atara [Builder]: Updating {name}...")
        run_cmd(f"cd {target_dir} && git pull", allow_fail=True)


# -------------------------------------------------------------------------
# 1. Install ComfyUI + custom nodes
# -------------------------------------------------------------------------
print("\n>>> 1. 基礎設施檢查 / Installing ComfyUI <<<")

run_cmd("pkill -f main.py || true", "清理舊 ComfyUI 進程...")
run_cmd("pkill -f cloudflared || true", "清理舊 Cloudflare tunnel...")

print("Atara [Builder]: Installing system packages...")
run_cmd("apt-get update && apt-get install -y libgl1 libglib2.0-0 ffmpeg zstd pciutils git")

clone_or_update(
    "https://github.com/comfyanonymous/ComfyUI.git",
    COMFY_DIR,
    "ComfyUI"
)

custom_nodes_dir = os.path.join(COMFY_DIR, "custom_nodes")
os.makedirs(custom_nodes_dir, exist_ok=True)

custom_nodes = [
    (
        "https://github.com/ltdrdata/ComfyUI-Manager.git",
        os.path.join(custom_nodes_dir, "ComfyUI-Manager"),
        "ComfyUI-Manager"
    ),
    (
        "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
        os.path.join(custom_nodes_dir, "ComfyUI-Impact-Pack"),
        "Impact Pack"
    ),
    (
        "https://github.com/kijai/ComfyUI-KJNodes.git",
        os.path.join(custom_nodes_dir, "ComfyUI-KJNodes"),
        "KJNodes"
    ),
    (
        "https://github.com/crystian/ComfyUI-Crystools.git",
        os.path.join(custom_nodes_dir, "ComfyUI-Crystools"),
        "Crystools GPU/CPU monitor"
    ),
    (
        "https://github.com/city96/ComfyUI-GGUF.git",
        os.path.join(custom_nodes_dir, "ComfyUI-GGUF"),
        "ComfyUI-GGUF"
    ),
    (
        "https://github.com/kijai/ComfyUI-WanVideoWrapper.git",
        os.path.join(custom_nodes_dir, "ComfyUI-WanVideoWrapper"),
        "ComfyUI-WanVideoWrapper"
    ),
]

for repo_url, target_dir, name in custom_nodes:
    clone_or_update(repo_url, target_dir, name)

print("Atara [Builder]: Installing ComfyUI requirements...")
run_cmd(f"pip install -r {COMFY_DIR}/requirements.txt")

requirements_files = [
    f"{custom_nodes_dir}/ComfyUI-Manager/requirements.txt",
    f"{custom_nodes_dir}/ComfyUI-Impact-Pack/requirements.txt",
    f"{custom_nodes_dir}/ComfyUI-KJNodes/requirements.txt",
    f"{custom_nodes_dir}/ComfyUI-Crystools/requirements.txt",
    f"{custom_nodes_dir}/ComfyUI-GGUF/requirements.txt",
    f"{custom_nodes_dir}/ComfyUI-WanVideoWrapper/requirements.txt",
]

for req in requirements_files:
    if os.path.exists(req):
        print(f"Atara [Builder]: Installing requirements: {req}")
        run_cmd(f"pip install -r {req}", allow_fail=True)
    else:
        print(f"Atara [Builder]: No requirements file found: {req}")

print("Atara [Builder]: Installing extra packages...")
run_cmd(
    "pip install comfy-cli torchsde einops transformers safetensors aiohttp "
    "accelerate pyyaml opencv-python matplotlib pillow scipy imageio[ffmpeg] "
    "moviepy huggingface_hub gguf"
)

# -------------------------------------------------------------------------
# 2. Link models from 646final
# -------------------------------------------------------------------------
print("\n>>> 2. 掛載 646final 模型 <<<")

mapping_list = [
    # --------------------------------------------------
    # Wan 2.2 I2V A14B normal GGUF models
    # --------------------------------------------------
    ("Wan2.2-I2V-A14B-HighNoise-Q8_0.gguf", "unet"),
    ("Wan2.2-I2V-A14B-LowNoise-Q8_0.gguf", "unet"),

    # Wan support files
    ("Wan2.1_VAE.safetensors", "vae"),
    ("Wan2.2_VAE.safetensors", "vae"),
    ("umt5-xxl-enc-fp8_e4m3fn.safetensors", "clip"),
    ("open-clip-xlm-roberta-large-vit-huge-14_visual_fp16.safetensors", "clip_vision"),
    ("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "clip"),
    ("detailz-wan.safetensors", "loras"),
    # --------------------------------------------------
    # Qwen Image Edit
    # --------------------------------------------------
    ("Qwen-Rapid-AIO-NSFW-v23.safetensors", "unet"),
    ("pig_qwen_image_vae_fp32-f16.gguf", "vae"),
    ("edit_0928_lora_step40000.safetensors", "loras"),

    # --------------------------------------------------
    # TurboWan files
    # Note: these are not normal WanVideoWrapper models.
    # Keep linked only if you have a TurboWan-specific workflow/loader.
    # --------------------------------------------------
    ("TurboWan2.2-I2V-A14B-high-720P-quant.pth", "unet"),
    ("TurboWan2.2-I2V-A14B-low-720P-quant.pth", "unet"),
]

for filename, target_sub in mapping_list:
    src = os.path.join(MODEL_SOURCE_PATH, filename)
    dst_dir = os.path.join(MODELS_DIR, target_sub)
    dst = os.path.join(dst_dir, filename)

    os.makedirs(dst_dir, exist_ok=True)

    if os.path.exists(src):
        if os.path.exists(dst) or os.path.islink(dst):
            os.remove(dst)
        os.symlink(src, dst)
        print(f"Atara [Link]: ✅ {filename} -> models/{target_sub}")
    else:
        print(f"Atara [Warning]: ❌ Missing: {filename}")

print("\nAtara [Check]: Current linked model folders:")
run_cmd(f"find {MODELS_DIR} -maxdepth 2 -type l -printf '%p -> %l\\n'", allow_fail=True)

# -------------------------------------------------------------------------
# 3. Start ComfyUI + Cloudflare tunnel
# -------------------------------------------------------------------------
print("\n>>> 3. 啟動 ComfyUI <<<")

comfyui_process = None
cloudflared_process = None


def wait_for_comfyui(port=8188, timeout=180):
    print("Atara [Network]: 等待 ComfyUI 啟動...")
    start = time.time()

    while time.time() - start < timeout:
        try:
            s = socket.create_connection(("localhost", port), timeout=2)
            s.close()
            print("Atara [Network]: ✅ ComfyUI 已啟動")
            return True
        except Exception:
            time.sleep(1)

    print("Atara [Network]: ❌ ComfyUI 啟動超時")
    return False


def start_comfyui():
    global comfyui_process

    print("Atara [Backend]: 啟動 ComfyUI...")
    cmd = "python main.py --listen 0.0.0.0 --port 8188 --preview-method auto"

    comfyui_process = subprocess.Popen(
        cmd,
        shell=True,
        cwd=COMFY_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        universal_newlines=True
    )

    for line in iter(comfyui_process.stdout.readline, ""):
        line = line.strip()
        if not line:
            continue

        # Print useful startup info
        if (
            "To see the GUI" in line
            or "Starting server" in line
            or "Total VRAM" in line
            or "Set vram state" in line
            or "CUDA" in line
            or "Crystools" in line
            or "WanVideo" in line
            or "GGUF" in line
        ):
            print(f"[ComfyUI]: {line}")

        # Print errors
        elif "Error" in line or "Traceback" in line or "Exception" in line:
            print(f"\033[1;31m[ComfyUI Error]: {line}\033[0m")


def start_cloudflare():
    global cloudflared_process

    print("Atara [Network]: 準備啟動 Cloudflare Tunnel...")

    if not wait_for_comfyui():
        return

    run_cmd("rm -f cloudflared")
    run_cmd(
        "curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
        "-o cloudflared"
    )
    run_cmd("chmod +x cloudflared")

    cmd = "./cloudflared tunnel --no-autoupdate --protocol http2 --url http://localhost:8188"

    cloudflared_process = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        universal_newlines=True
    )

    printed_url = False

    for line in iter(cloudflared_process.stdout.readline, ""):
        line = line.strip()
        if not line:
            continue

        print(f"[cloudflared] {line}")

        if "trycloudflare.com" in line and not printed_url:
            match = re.search(r"(https?://[^\s]+)", line)
            if match:
                printed_url = True
                print("\n==========================================================")
                print(f"   >>> 🌐 ComfyUI Public Link: {match.group(1)} <<<")
                print("==========================================================\n")


# -------------------------------------------------------------------------
# 4. Run threads
# -------------------------------------------------------------------------
t_comfy = threading.Thread(target=start_comfyui, daemon=True)
t_tunnel = threading.Thread(target=start_cloudflare, daemon=True)

t_comfy.start()
time.sleep(3)
t_tunnel.start()

try:
    while True:
        time.sleep(10)

        if comfyui_process and comfyui_process.poll() is not None:
            print(f"Atara [CRITICAL]: ComfyUI stopped. Code: {comfyui_process.returncode}")
            break

except KeyboardInterrupt:
    print("\nAtara: 使用者中斷，開始關閉服務...")

finally:
    if cloudflared_process and cloudflared_process.poll() is None:
        cloudflared_process.terminate()
        print("Atara [Cleanup]: Cloudflare tunnel terminated")

    if comfyui_process and comfyui_process.poll() is None:
        comfyui_process.terminate()
        print("Atara [Cleanup]: ComfyUI terminated")

    print("Atara: 全部程序已嘗試關閉，結束。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Atara [Groq]: ✅ GROQ_API_KEY loaded from Colab secrets

>>> 1. 基礎設施檢查 / Installing ComfyUI <<<
Atara [System]: 清理舊 ComfyUI 進程...
Atara [System]: 清理舊 Cloudflare tunnel...
Atara [Builder]: Installing system packages...
Atara [Builder]: Cloning ComfyUI...
Atara [Builder]: Cloning ComfyUI-Manager...
Atara [Builder]: Cloning Impact Pack...
Atara [Builder]: Cloning KJNodes...
Atara [Builder]: Cloning Crystools GPU/CPU monitor...
Atara [Builder]: Cloning ComfyUI-GGUF...
Atara [Builder]: Cloning ComfyUI-WanVideoWrapper...
Atara [Builder]: Installing ComfyUI requirements...
Atara [Builder]: Installing requirements: /content/ComfyUI/custom_nodes/ComfyUI-Manager/requirements.txt
Atara [Builder]: Installing requirements: /content/ComfyUI/custom_nodes/ComfyUI-Impact-Pack/requirements.txt
Atara [Builder]: Installing requirements: /content/ComfyUI/custom_nodes/ComfyUI-KJNod